# Revision 09 — Condition influence and optimization-seed stability at K=46

This notebook addresses two reviewer-facing questions for the representative
across-condition MS-AA solution.

**Recommended repository location:**

`analysis/04_interpretation/revision_09_condition_influence_and_seed_stability_K46.ipynb`

Execute the notebook from the repository root so all project-relative paths resolve correctly.

## Important correction about the generator matrix \(C\)

In this implementation, \(C\) is **not indexed by participants or experimental
conditions**. Its shape is \((sV,K)\), where \(sV\) is the reduced feature
dimension. For spatial MS-AA at \(K=46\), \(C\) is therefore \(700\times46\):
its rows correspond to HTFA-node generator weights. The fraction of \(C\)
attributable to intact, word-scrambled, or rest is not mathematically defined.

The notebook therefore separates three questions:

1. **Generator concentration:** Is each selected archetype generated from a
   broad or concentrated subset of the 700 feature dimensions?
2. **Condition-specific deployment:** Is each selected archetype expressed
   disproportionately strongly in one condition in the fitted model?
   This is descriptive and is not equivalent to condition influence on fitting.
3. **Condition influence on formation:** Does the fitted solution change when
   one condition is omitted? This requires leave-one-condition-out refits.
4. **Optimization stability:** Are the same archetypes, reconstructions, and
   descriptive network profiles recovered across independently initialized
   fits after permutation alignment?

The first two sections run from the existing \(K=46\) fit. The latter sections
run automatically when the corresponding optional fit files are provided.


In [ ]:
# ============================================================
# SETTINGS
# ============================================================

from pathlib import Path

ANALYSIS_TYPE = "spatial"
K = 46
CONDITIONS = ["intact", "word", "rest"]

CANONICAL_FIT = Path(
    "msaa_flexible_outputs_npz/"
    "spatialAA_across_acrossCond_K46.npz"
)
import sys
sys.path.insert(0, "src")
from msaa_pipeline.paper_archetypes import paper_archetypes
# M0-M4, in the K=46 fit's own numbering (canonical 28,35,16,39,45; legacy 4,5,7,8,9)
SELECTED_ARCHETYPES, ARCHETYPE_LABELS = paper_archetypes(CANONICAL_FIT)
LABEL = dict(zip(SELECTED_ARCHETYPES, ARCHETYPE_LABELS))

OUTPUT_DIR = Path(
    "outputs/interpretation/"
    "condition_influence_and_seed_stability_K46"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Optional leave-one-condition-out fits
#
# Create these with exactly the same preprocessing, K, fitting
# options, and random seed as the canonical fit, changing only
# the set of conditions supplied to the fitter.
# ------------------------------------------------------------

LEAVE_ONE_CONDITION_OUT_FITS = {
    "drop_intact": Path(
        "condition_influence_fits/"
        "spatialAA_across_drop_intact_K46.npz"
    ),
    "drop_word": Path(
        "condition_influence_fits/"
        "spatialAA_across_drop_word_K46.npz"
    ),
    "drop_rest": Path(
        "condition_influence_fits/"
        "spatialAA_across_drop_rest_K46.npz"
    ),
}

# ------------------------------------------------------------
# Optional independently initialized full fits
#
# Either list files explicitly or give a recursive glob.
# The canonical fit is automatically excluded.
# ------------------------------------------------------------

SEED_FIT_FILES = [
    # Path("seeds/full/seed01/msaa_flexible_outputs_npz/spatialAA_across_acrossCond_K46.npz"),
]

# Default: the per-seed fits written by analysis/08_multiseed_stability/run_local.py
SEED_FIT_GLOB = (
    "seeds/full/seed*/msaa_flexible_outputs_npz/"
    "spatialAA_across_acrossCond_K46.npz"
)

# Optional audited node-to-network mapping from Figure 5.
NODE_AUDIT_CSV = Path(
    "paper_figure_outputs/"
    "Figure5_HTFA_Yeo_mapping_audit_nodes.csv"
)

MAX_LAG_TR = 2
EPS = 1e-12

print("Canonical fit:", CANONICAL_FIT)
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
# ============================================================
# IMPORTS AND GENERAL HELPERS
# ============================================================

import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from IPython.display import display


def to_float_array(x):
    return np.asarray(x, dtype=float)


def normalize_labels(values):
    out = []
    for value in np.asarray(values).reshape(-1):
        if isinstance(value, bytes):
            out.append(value.decode("utf-8"))
        else:
            out.append(str(value))
    return np.asarray(out, dtype=object)


def load_fit(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    data = np.load(path, allow_pickle=True)

    results_subj = data["results_subj"].tolist()
    if isinstance(results_subj, np.ndarray):
        results_subj = results_subj.tolist()

    out = {
        "path": path,
        "K": int(np.asarray(data["K"]).item()),
        "results_subj": results_subj,
        "condition_labels_str": normalize_labels(
            data["condition_labels_str"]
        ),
    }

    if "C" in data:
        out["C"] = np.asarray(data["C"], dtype=float)
    else:
        out["C"] = None

    for key in [
        "analysis_type",
        "fit_scope",
        "cost_fun",
        "varexpl",
    ]:
        if key in data:
            out[key] = data[key]

    return out


def safe_corr(a, b):
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()

    valid = np.isfinite(a) & np.isfinite(b)
    a = a[valid]
    b = b[valid]

    if len(a) < 3:
        return np.nan

    a = a - a.mean()
    b = b - b.mean()

    denom = np.linalg.norm(a) * np.linalg.norm(b)

    if denom <= EPS:
        return np.nan

    return float(np.dot(a, b) / denom)


def best_lag_corr(reference, candidate, max_lag=2):
    reference = np.asarray(reference, dtype=float).ravel()
    candidate = np.asarray(candidate, dtype=float).ravel()

    if len(reference) != len(candidate):
        raise ValueError(
            "Lag comparison requires equal-length vectors."
        )

    rows = []

    for lag in range(-int(max_lag), int(max_lag) + 1):
        if lag < 0:
            x = reference[-lag:]
            y = candidate[:lag]
        elif lag > 0:
            x = reference[:-lag]
            y = candidate[lag:]
        else:
            x = reference
            y = candidate

        rows.append(
            {
                "lag": lag,
                "correlation": safe_corr(x, y),
            }
        )

    lag_df = pd.DataFrame(rows)
    valid = lag_df["correlation"].notna()

    if not valid.any():
        return {
            "best_lag": np.nan,
            "best_lag_correlation": np.nan,
            "zero_lag_correlation": np.nan,
        }

    best_i = lag_df.loc[
        valid,
        "correlation",
    ].idxmax()

    zero = lag_df.loc[
        lag_df["lag"] == 0,
        "correlation",
    ].iloc[0]

    return {
        "best_lag": int(lag_df.loc[best_i, "lag"]),
        "best_lag_correlation": float(
            lag_df.loc[best_i, "correlation"]
        ),
        "zero_lag_correlation": float(zero),
    }


def cosine_similarity_columns(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    if A.shape[0] != B.shape[0]:
        raise ValueError(
            f"Generator row mismatch: {A.shape} vs {B.shape}"
        )

    An = A / (
        np.linalg.norm(A, axis=0, keepdims=True) + EPS
    )
    Bn = B / (
        np.linalg.norm(B, axis=0, keepdims=True) + EPS
    )

    return An.T @ Bn


def hungarian_match_generators(reference_C, candidate_C):
    similarity = cosine_similarity_columns(
        reference_C,
        candidate_C,
    )

    ref_idx, candidate_idx = linear_sum_assignment(
        -similarity
    )

    mapping = {
        int(r): int(c)
        for r, c in zip(ref_idx, candidate_idx)
    }

    rows = [
        {
            "reference_archetype": int(r),
            "candidate_archetype": int(c),
            "generator_cosine": float(similarity[r, c]),
        }
        for r, c in zip(ref_idx, candidate_idx)
    ]

    return mapping, pd.DataFrame(rows), similarity


def condition_subjects(fit, condition):
    labels = fit["condition_labels_str"]
    idx = np.where(labels == str(condition))[0]

    return [
        fit["results_subj"][int(i)]
        for i in idx
    ]


def mean_archetype_vectors(fit, condition, archetype):
    subjects = condition_subjects(fit, condition)

    if not subjects:
        return None

    primary = np.stack(
        [
            to_float_array(sub["sXC"])[:, int(archetype)]
            for sub in subjects
        ],
        axis=0,
    ).mean(axis=0)

    deployment = np.stack(
        [
            to_float_array(sub["S"])[int(archetype), :]
            for sub in subjects
        ],
        axis=0,
    ).mean(axis=0)

    if ANALYSIS_TYPE == "spatial":
        temporal = primary
        spatial = deployment
    elif ANALYSIS_TYPE == "temporal":
        spatial = primary
        temporal = deployment
    else:
        raise ValueError(ANALYSIS_TYPE)

    return {
        "primary": primary,
        "deployment": deployment,
        "temporal": temporal,
        "spatial": spatial,
    }


def reconstruct_subject(sub):
    sXC = to_float_array(sub["sXC"])
    S = to_float_array(sub["S"])

    if ANALYSIS_TYPE == "spatial":
        return sXC @ S

    if ANALYSIS_TYPE == "temporal":
        return (sXC @ S).T

    raise ValueError(ANALYSIS_TYPE)


def effective_number(values):
    values = np.asarray(values, dtype=float)
    values = np.nan_to_num(
        np.abs(values),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    total = values.sum()

    if total <= 0:
        return {
            "max_share": np.nan,
            "effective_n": np.nan,
            "effective_fraction": np.nan,
        }

    shares = values / total

    return {
        "max_share": float(shares.max()),
        "effective_n": float(
            1.0 / np.sum(shares ** 2)
        ),
        "effective_fraction": float(
            1.0 / np.sum(shares ** 2) / len(shares)
        ),
    }


## 1. Load the canonical fit and verify the meaning of \(C\)

For this implementation, `C.shape == (sV, K)`. In spatial MS-AA,
\(sV=700\), so the rows of \(C\) index HTFA feature dimensions, not
participant-condition observations. Each column is nonnegative and sums to one.


In [ ]:
canonical = load_fit(CANONICAL_FIT)

if canonical["K"] != K:
    raise ValueError(
        f"Expected K={K}, found K={canonical['K']}"
    )

C = canonical["C"]

if C is None:
    raise ValueError(
        "The canonical NPZ does not contain C."
    )

print("C shape:", C.shape)
print("Number of fit entries:", len(canonical["results_subj"]))

display(
    pd.Series(
        canonical["condition_labels_str"]
    )
    .value_counts()
    .reindex(CONDITIONS)
    .to_frame("n_observations")
)

column_sums = C.sum(axis=0)

print(
    "C minimum:",
    float(np.nanmin(C)),
)
print(
    "Maximum absolute deviation of column sum from 1:",
    float(np.nanmax(np.abs(column_sums - 1.0))),
)

if C.shape != (700, K) and ANALYSIS_TYPE == "spatial":
    warnings.warn(
        f"Expected C shape (700,{K}) for this spatial fit; "
        f"found {C.shape}."
    )


## 2. Generator concentration across feature dimensions

This section is a literal analysis of \(C\). It answers whether each archetype
is generated from a broad or concentrated subset of the feature dimensions.
It does **not** measure condition contribution.


In [ ]:
generator_rows = []

for k in SELECTED_ARCHETYPES:
    weights = np.asarray(C[:, int(k)], dtype=float)
    weights = np.clip(weights, 0.0, None)

    total = weights.sum()
    shares = weights / (total + EPS)

    entropy = (
        -np.sum(
            shares * np.log(shares + EPS)
        )
        / np.log(len(shares))
    )

    stats = effective_number(weights)

    generator_rows.append(
        {
            "archetype": int(k),
            "n_generator_dimensions": len(weights),
            "max_generator_share": stats["max_share"],
            "effective_generator_n": stats["effective_n"],
            "effective_generator_fraction": (
                stats["effective_fraction"]
            ),
            "normalized_generator_entropy": float(entropy),
        }
    )

generator_summary_df = pd.DataFrame(generator_rows)

display(generator_summary_df)

generator_summary_df.to_csv(
    OUTPUT_DIR / "selected_archetype_C_generator_concentration.csv",
    index=False,
)


## 3. Descriptive condition-specific deployment

For each participant-condition observation and selected archetype, the exact
Frobenius magnitude of the rank-one contribution is

\[
\|sXC_{:,k}\|_2\,\|S_{k,:}\|_2.
\]

The resulting condition shares describe how strongly a fitted archetype is
deployed in each condition. They do not establish how much each condition
caused the shared basis to take its fitted form.

No cross-condition paired tests are performed because participant identifiers
are unavailable in the processed fit.


In [ ]:
deployment_rows = []

for observation_index, (sub, condition) in enumerate(
    zip(
        canonical["results_subj"],
        canonical["condition_labels_str"],
    )
):
    sXC = to_float_array(sub["sXC"])
    S = to_float_array(sub["S"])

    for k in SELECTED_ARCHETYPES:
        primary = sXC[:, int(k)]
        deployment = S[int(k), :]

        primary_l2 = np.linalg.norm(primary)
        deployment_l2 = np.linalg.norm(deployment)

        deployment_rows.append(
            {
                "observation_index": int(observation_index),
                "condition": str(condition),
                "archetype": int(k),
                "primary_l2": float(primary_l2),
                "deployment_l1": float(
                    np.sum(np.abs(deployment))
                ),
                "deployment_l2": float(deployment_l2),
                "rank1_frobenius": float(
                    primary_l2 * deployment_l2
                ),
            }
        )

deployment_df = pd.DataFrame(deployment_rows)

condition_deployment_df = (
    deployment_df
    .groupby(
        ["archetype", "condition"],
        as_index=False,
    )
    .agg(
        n_observations=("rank1_frobenius", "size"),
        mean_rank1_frobenius=("rank1_frobenius", "mean"),
        median_rank1_frobenius=("rank1_frobenius", "median"),
        sd_rank1_frobenius=("rank1_frobenius", "std"),
        mean_deployment_l2=("deployment_l2", "mean"),
        mean_primary_l2=("primary_l2", "mean"),
    )
)

condition_deployment_df["condition_share"] = (
    condition_deployment_df["mean_rank1_frobenius"]
    / condition_deployment_df
      .groupby("archetype")["mean_rank1_frobenius"]
      .transform("sum")
)

condition_concentration_rows = []

for k in SELECTED_ARCHETYPES:
    cur = condition_deployment_df[
        condition_deployment_df["archetype"] == int(k)
    ].set_index("condition").reindex(CONDITIONS)

    shares = cur["condition_share"].to_numpy(dtype=float)

    normalized_entropy = (
        -np.nansum(
            shares * np.log(shares + EPS)
        )
        / np.log(len(CONDITIONS))
    )

    condition_concentration_rows.append(
        {
            "archetype": int(k),
            "max_condition_share": float(np.nanmax(shares)),
            "normalized_condition_entropy": float(
                normalized_entropy
            ),
            "l1_distance_from_equal_thirds": float(
                np.nansum(
                    np.abs(
                        shares - (1.0 / len(CONDITIONS))
                    )
                )
            ),
        }
    )

condition_concentration_df = pd.DataFrame(
    condition_concentration_rows
)

observation_breadth_rows = []

for k in SELECTED_ARCHETYPES:
    for condition in CONDITIONS:
        values = deployment_df.loc[
            (deployment_df["archetype"] == int(k))
            & (deployment_df["condition"] == condition),
            "rank1_frobenius",
        ].to_numpy()

        stats = effective_number(values)

        observation_breadth_rows.append(
            {
                "archetype": int(k),
                "condition": condition,
                "n_observations": len(values),
                **stats,
            }
        )

observation_breadth_df = pd.DataFrame(
    observation_breadth_rows
)

display(
    condition_deployment_df.sort_values(
        ["archetype", "condition"]
    )
)

display(condition_concentration_df)
display(observation_breadth_df)

deployment_df.to_csv(
    OUTPUT_DIR / "selected_archetype_deployment_by_observation.csv",
    index=False,
)

condition_deployment_df.to_csv(
    OUTPUT_DIR / "selected_archetype_deployment_by_condition.csv",
    index=False,
)

condition_concentration_df.to_csv(
    OUTPUT_DIR / "selected_archetype_condition_concentration.csv",
    index=False,
)

observation_breadth_df.to_csv(
    OUTPUT_DIR / "selected_archetype_observation_breadth_by_condition.csv",
    index=False,
)


In [ ]:
# Stacked descriptive condition shares.

share_wide = (
    condition_deployment_df
    .pivot(
        index="archetype",
        columns="condition",
        values="condition_share",
    )
    .reindex(
        index=SELECTED_ARCHETYPES,
        columns=CONDITIONS,
    )
)

ax = share_wide.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
)

ax.set_xlabel("Selected archetype")
ax.set_ylabel("Fraction of mean rank-1 contribution")
ax.set_title(
    "Descriptive condition deployment of selected archetypes"
)
ax.legend(
    title="Condition",
    frameon=False,
)

plt.tight_layout()

condition_share_png = (
    OUTPUT_DIR
    / "selected_archetype_condition_deployment.png"
)
condition_share_pdf = (
    OUTPUT_DIR
    / "selected_archetype_condition_deployment.pdf"
)

plt.savefig(
    condition_share_png,
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    condition_share_pdf,
    bbox_inches="tight",
)
plt.show()

print("Saved:", condition_share_png)
print("Saved:", condition_share_pdf)


## 4. Leave-one-condition-out influence on archetype formation

This is the direct sensitivity analysis for the reviewer's question.

Create three additional \(K=46\) spatial across-condition fits using the same
preprocessing and fitting options as the canonical fit:

- `drop_intact`: fit word-scrambled + rest;
- `drop_word`: fit intact + rest;
- `drop_rest`: fit intact + word-scrambled.

The fitted generator matrices are aligned to the canonical \(C\) using
one-to-one Hungarian matching. The notebook then compares:

- generator-column cosine similarity;
- temporal-motif correlation at zero lag and best lag within \(\pm2\) TRs;
- spatial-deployment-map correlation.

If these files are absent, this section prints the exact missing paths and
continues.


In [ ]:
# ============================================================
# LEAVE-ONE-CONDITION-OUT COMPARISON
# ============================================================

loo_rows = []
loo_match_rows = []

for label, path in LEAVE_ONE_CONDITION_OUT_FITS.items():
    path = Path(path)

    if not path.exists():
        print("Missing optional LOO fit:", path)
        continue

    candidate = load_fit(path)

    if candidate["K"] != K:
        raise ValueError(
            f"{path}: expected K={K}, found {candidate['K']}"
        )

    if candidate["C"] is None:
        raise ValueError(
            f"{path}: no C matrix found."
        )

    mapping, match_df, _ = hungarian_match_generators(
        canonical["C"],
        candidate["C"],
    )

    match_df["comparison"] = label
    match_df["fit_file"] = str(path)
    loo_match_rows.append(match_df)

    common_conditions = [
        condition
        for condition in CONDITIONS
        if (
            condition
            in set(candidate["condition_labels_str"])
            and condition
            in set(canonical["condition_labels_str"])
        )
    ]

    for reference_k in range(K):
        candidate_k = mapping[int(reference_k)]

        generator_cosine = match_df.loc[
            match_df["reference_archetype"] == int(reference_k),
            "generator_cosine",
        ].iloc[0]

        condition_metric_rows = []

        for condition in common_conditions:
            ref_vecs = mean_archetype_vectors(
                canonical,
                condition,
                reference_k,
            )
            cand_vecs = mean_archetype_vectors(
                candidate,
                condition,
                candidate_k,
            )

            if ref_vecs is None or cand_vecs is None:
                continue

            temporal_lag = best_lag_corr(
                ref_vecs["temporal"],
                cand_vecs["temporal"],
                max_lag=MAX_LAG_TR,
            )

            condition_metric_rows.append(
                {
                    "condition": condition,
                    "spatial_map_correlation": safe_corr(
                        ref_vecs["spatial"],
                        cand_vecs["spatial"],
                    ),
                    **temporal_lag,
                }
            )

        condition_metric_df = pd.DataFrame(
            condition_metric_rows
        )

        loo_rows.append(
            {
                "comparison": label,
                "fit_file": str(path),
                "reference_archetype": int(reference_k),
                "candidate_archetype": int(candidate_k),
                "is_selected_reference": bool(
                    reference_k in SELECTED_ARCHETYPES
                ),
                "n_common_conditions": len(condition_metric_df),
                "generator_cosine": float(generator_cosine),
                "mean_spatial_map_correlation": (
                    condition_metric_df[
                        "spatial_map_correlation"
                    ].mean()
                    if len(condition_metric_df)
                    else np.nan
                ),
                "mean_temporal_zero_lag_correlation": (
                    condition_metric_df[
                        "zero_lag_correlation"
                    ].mean()
                    if len(condition_metric_df)
                    else np.nan
                ),
                "mean_temporal_best_lag_correlation": (
                    condition_metric_df[
                        "best_lag_correlation"
                    ].mean()
                    if len(condition_metric_df)
                    else np.nan
                ),
                "mean_absolute_best_lag_TR": (
                    condition_metric_df[
                        "best_lag"
                    ].abs().mean()
                    if len(condition_metric_df)
                    else np.nan
                ),
            }
        )

if loo_rows:
    loo_comparison_df = pd.DataFrame(loo_rows)
    loo_matches_df = pd.concat(
        loo_match_rows,
        ignore_index=True,
    )

    display(
        loo_comparison_df[
            loo_comparison_df["is_selected_reference"]
        ].sort_values(
            ["comparison", "reference_archetype"]
        )
    )

    loo_summary_df = (
        loo_comparison_df
        .groupby(
            ["comparison", "is_selected_reference"],
            as_index=False,
        )
        .agg(
            n_archetypes=("reference_archetype", "size"),
            mean_generator_cosine=("generator_cosine", "mean"),
            min_generator_cosine=("generator_cosine", "min"),
            mean_spatial_map_correlation=(
                "mean_spatial_map_correlation",
                "mean",
            ),
            mean_temporal_zero_lag_correlation=(
                "mean_temporal_zero_lag_correlation",
                "mean",
            ),
            mean_temporal_best_lag_correlation=(
                "mean_temporal_best_lag_correlation",
                "mean",
            ),
        )
    )

    display(loo_summary_df)

    loo_comparison_df.to_csv(
        OUTPUT_DIR / "leave_one_condition_out_archetype_comparison.csv",
        index=False,
    )

    loo_matches_df.to_csv(
        OUTPUT_DIR / "leave_one_condition_out_hungarian_matches.csv",
        index=False,
    )

    loo_summary_df.to_csv(
        OUTPUT_DIR / "leave_one_condition_out_summary.csv",
        index=False,
    )
else:
    print(
        "\nNo leave-one-condition-out fits were found. "
        "The descriptive sections above are complete, but a direct "
        "condition-formation claim should wait for these refits."
    )


## 5. Optimization-seed stability after permutation alignment

The same analysis can be applied to Ben's independently initialized \(K=46\)
fits. Archetype indices are arbitrary, so every fit is first aligned to the
canonical fit using Hungarian matching on the shared generator columns.

For each matched archetype, the notebook reports:

- generator cosine similarity;
- spatial-map correlation;
- temporal zero-lag correlation;
- best temporal correlation allowing \(\pm2\) TR;
- best lag;
- full-reconstruction agreement by participant-condition observation.

If the audited Figure 5 node-to-network table is available, it additionally
compares the descriptive Yeo-7 profile of each matched selected archetype and
the aggregate selected-five profile.

This tests reproducibility of the fitted motifs. Independently selected top-5
membership should be added after the seed-specific participant-held-out ranking
files become available.


In [ ]:
# ============================================================
# DISCOVER OPTIONAL SEED FITS
# ============================================================

seed_paths = {
    Path(path).resolve()
    for path in SEED_FIT_FILES
    if Path(path).exists()
}

if SEED_FIT_GLOB:
    for path in Path(".").glob(SEED_FIT_GLOB):
        if path.is_file():
            seed_paths.add(path.resolve())

canonical_resolved = CANONICAL_FIT.resolve()

seed_paths = sorted(
    path
    for path in seed_paths
    if path != canonical_resolved
)

print("Discovered seed fits:", len(seed_paths))

for path in seed_paths:
    print(" -", path)


In [ ]:
# ============================================================
# OPTIONAL NETWORK-MAPPING INPUT
# ============================================================

network_labels = None
NETWORK_ORDER = [
    "Visual",
    "Somatomotor",
    "Dorsal attention",
    "Ventral attention",
    "Limbic",
    "Frontoparietal",
    "Default mode",
]

if NODE_AUDIT_CSV.exists():
    node_audit = (
        pd.read_csv(NODE_AUDIT_CSV)
        .sort_values("node")
    )

    network_labels = node_audit[
        "assigned_network_integrated_overlap"
    ].astype(str).to_numpy()

    if len(network_labels) != C.shape[0]:
        warnings.warn(
            "Node-audit length does not match C rows; "
            "network stability will be skipped."
        )
        network_labels = None

    print("Loaded network mapping:", NODE_AUDIT_CSV)
else:
    print(
        "Optional node audit not found; "
        "network-profile stability will be skipped."
    )


def network_profile(spatial_map):
    if network_labels is None:
        return None

    values = np.abs(
        np.asarray(spatial_map, dtype=float)
    )

    total = values.sum()

    if total <= 0:
        return np.full(
            len(NETWORK_ORDER),
            np.nan,
        )

    return np.asarray(
        [
            values[
                network_labels == network
            ].sum() / total
            for network in NETWORK_ORDER
        ],
        dtype=float,
    )


In [ ]:
# ============================================================
# RUN SEED-STABILITY ANALYSIS
# ============================================================

seed_archetype_rows = []
seed_reconstruction_rows = []
seed_network_rows = []

canonical_condition_vectors = {
    condition: {
        k: mean_archetype_vectors(
            canonical,
            condition,
            k,
        )
        for k in range(K)
    }
    for condition in CONDITIONS
}

canonical_selected_profiles = {}

if network_labels is not None:
    for k in SELECTED_ARCHETYPES:
        pooled_spatial = np.stack(
            [
                to_float_array(sub["S"])[int(k), :]
                for sub in canonical["results_subj"]
            ],
            axis=0,
        ).mean(axis=0)

        canonical_selected_profiles[int(k)] = network_profile(
            pooled_spatial
        )

for seed_number, path in enumerate(seed_paths):
    candidate = load_fit(path)

    if candidate["K"] != K:
        warnings.warn(
            f"Skipping {path}: K={candidate['K']} rather than {K}."
        )
        continue

    if candidate["C"] is None:
        warnings.warn(
            f"Skipping {path}: no C matrix."
        )
        continue

    mapping, match_df, _ = hungarian_match_generators(
        canonical["C"],
        candidate["C"],
    )

    seed_label = path.parent.parent.name if path.parent.name == "msaa_flexible_outputs_npz" else path.parent.name

    for reference_k in range(K):
        candidate_k = mapping[int(reference_k)]

        generator_cosine = match_df.loc[
            match_df["reference_archetype"] == int(reference_k),
            "generator_cosine",
        ].iloc[0]

        per_condition_rows = []

        for condition in CONDITIONS:
            ref_vecs = canonical_condition_vectors[
                condition
            ][reference_k]

            cand_vecs = mean_archetype_vectors(
                candidate,
                condition,
                candidate_k,
            )

            if ref_vecs is None or cand_vecs is None:
                continue

            temporal_lag = best_lag_corr(
                ref_vecs["temporal"],
                cand_vecs["temporal"],
                max_lag=MAX_LAG_TR,
            )

            per_condition_rows.append(
                {
                    "condition": condition,
                    "spatial_map_correlation": safe_corr(
                        ref_vecs["spatial"],
                        cand_vecs["spatial"],
                    ),
                    **temporal_lag,
                }
            )

        metric_df = pd.DataFrame(
            per_condition_rows
        )

        row = {
            "seed_index": int(seed_number),
            "seed_label": seed_label,
            "fit_file": str(path),
            "reference_archetype": int(reference_k),
            "candidate_archetype": int(candidate_k),
            "is_selected_reference": bool(
                reference_k in SELECTED_ARCHETYPES
            ),
            "generator_cosine": float(generator_cosine),
            "mean_spatial_map_correlation": (
                metric_df[
                    "spatial_map_correlation"
                ].mean()
                if len(metric_df)
                else np.nan
            ),
            "mean_temporal_zero_lag_correlation": (
                metric_df[
                    "zero_lag_correlation"
                ].mean()
                if len(metric_df)
                else np.nan
            ),
            "mean_temporal_best_lag_correlation": (
                metric_df[
                    "best_lag_correlation"
                ].mean()
                if len(metric_df)
                else np.nan
            ),
            "mean_absolute_best_lag_TR": (
                metric_df[
                    "best_lag"
                ].abs().mean()
                if len(metric_df)
                else np.nan
            ),
        }

        if (
            network_labels is not None
            and reference_k in SELECTED_ARCHETYPES
        ):
            seed_spatial = np.stack(
                [
                    to_float_array(sub["S"])[candidate_k, :]
                    for sub in candidate["results_subj"]
                ],
                axis=0,
            ).mean(axis=0)

            seed_profile = network_profile(seed_spatial)
            reference_profile = canonical_selected_profiles[
                int(reference_k)
            ]

            row["network_profile_correlation"] = safe_corr(
                reference_profile,
                seed_profile,
            )

            row["max_absolute_network_change"] = float(
                np.nanmax(
                    np.abs(
                        seed_profile - reference_profile
                    )
                )
            )

            for network, mass in zip(
                NETWORK_ORDER,
                seed_profile,
            ):
                seed_network_rows.append(
                    {
                        "seed_index": int(seed_number),
                        "seed_label": seed_label,
                        "fit_file": str(path),
                        "reference_archetype": int(reference_k),
                        "candidate_archetype": int(candidate_k),
                        "network": network,
                        "mass_fraction": float(mass),
                    }
                )

        seed_archetype_rows.append(row)

    # Full-reconstruction agreement.
    #
    # This assumes that every independently initialized fit received the
    # observations in the same order. The condition-label sequence is checked.
    if np.array_equal(
        canonical["condition_labels_str"],
        candidate["condition_labels_str"],
    ):
        for observation_index, (
            ref_sub,
            cand_sub,
            condition,
        ) in enumerate(
            zip(
                canonical["results_subj"],
                candidate["results_subj"],
                canonical["condition_labels_str"],
            )
        ):
            ref_recon = reconstruct_subject(ref_sub)
            cand_recon = reconstruct_subject(cand_sub)

            seed_reconstruction_rows.append(
                {
                    "seed_index": int(seed_number),
                    "seed_label": seed_label,
                    "fit_file": str(path),
                    "observation_index": int(observation_index),
                    "condition": str(condition),
                    "reconstruction_correlation": safe_corr(
                        ref_recon,
                        cand_recon,
                    ),
                }
            )
    else:
        warnings.warn(
            f"{path}: condition-label order differs from canonical; "
            "entry-wise reconstruction agreement skipped."
        )

if seed_archetype_rows:
    seed_archetype_df = pd.DataFrame(
        seed_archetype_rows
    )

    seed_reconstruction_df = pd.DataFrame(
        seed_reconstruction_rows
    )

    display(
        seed_archetype_df[
            seed_archetype_df["is_selected_reference"]
        ].sort_values(
            ["seed_index", "reference_archetype"]
        )
    )

    seed_summary_df = (
        seed_archetype_df
        .groupby(
            ["seed_index", "seed_label", "is_selected_reference"],
            as_index=False,
        )
        .agg(
            n_archetypes=("reference_archetype", "size"),
            mean_generator_cosine=("generator_cosine", "mean"),
            min_generator_cosine=("generator_cosine", "min"),
            mean_spatial_map_correlation=(
                "mean_spatial_map_correlation",
                "mean",
            ),
            mean_temporal_zero_lag_correlation=(
                "mean_temporal_zero_lag_correlation",
                "mean",
            ),
            mean_temporal_best_lag_correlation=(
                "mean_temporal_best_lag_correlation",
                "mean",
            ),
            mean_absolute_best_lag_TR=(
                "mean_absolute_best_lag_TR",
                "mean",
            ),
        )
    )

    if len(seed_reconstruction_df):
        recon_summary = (
            seed_reconstruction_df
            .groupby(
                ["seed_index", "seed_label"],
                as_index=False,
            )
            .agg(
                mean_reconstruction_correlation=(
                    "reconstruction_correlation",
                    "mean",
                ),
                min_reconstruction_correlation=(
                    "reconstruction_correlation",
                    "min",
                ),
            )
        )

        seed_summary_df = seed_summary_df.merge(
            recon_summary,
            on=["seed_index", "seed_label"],
            how="left",
        )

    display(seed_summary_df)

    seed_archetype_df.to_csv(
        OUTPUT_DIR / "seed_aligned_archetype_stability.csv",
        index=False,
    )

    seed_summary_df.to_csv(
        OUTPUT_DIR / "seed_stability_summary.csv",
        index=False,
    )

    if len(seed_reconstruction_df):
        seed_reconstruction_df.to_csv(
            OUTPUT_DIR / "seed_reconstruction_agreement.csv",
            index=False,
        )

    if seed_network_rows:
        seed_network_df = pd.DataFrame(
            seed_network_rows
        )

        seed_network_df.to_csv(
            OUTPUT_DIR / "seed_aligned_network_profiles.csv",
            index=False,
        )
else:
    print(
        "\nNo optional seed fits were analyzed. "
        "Set SEED_FIT_FILES or SEED_FIT_GLOB and rerun."
    )


## 6. Interpretation and next step

### What can be reported immediately

- \(C\) is a feature-generator matrix, not a participant-by-archetype or
  condition-by-archetype matrix.
- Condition-specific deployment can be described using the observation-level
  rank-one contribution magnitudes.
- Generator concentration can be reported across the 700 HTFA feature
  dimensions.

### What is needed for a direct condition-formation statement

Run the three leave-one-condition-out \(K=46\) fits. A condition has strong
influence on formation if omitting it systematically lowers the aligned
generator, temporal-motif, or spatial-map similarity—especially for the five
held-out-selected archetypes.

### What is needed to finish optimization-stability reporting

Provide the ten independently initialized \(K=46\) fit files. The analysis
above aligns them to the canonical fit and measures motif, reconstruction, and
descriptive network-profile agreement.

After Ben completes the participant-held-out ranking for each fit seed, add:

- aligned top-5 selection frequency;
- Jaccard overlap of independently selected sets;
- agreement of held-out decoding curves across seeds.

Do not treat archetype numbers as intrinsic identities. Manuscript labels such
as A4 refer to the canonical fit; replicate fits are aligned to that reference.


## Running this notebook

From the repository root:

```bash
python scripts/execute_notebook.py \
  analysis/04_interpretation/revision_09_condition_influence_and_seed_stability_K46.ipynb
```

Generated tables and figures are written to:

```text
outputs/interpretation/condition_influence_and_seed_stability_K46/
```
